# Parkshare - Setup DB SQLite (KPI communes)

Notebook pour creer et alimenter la base SQLite KPI a partir de `data/Sources_clean/kpi_potentiel_communes.csv`.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().resolve().parent
SOURCES_CLEAN_DIR = ROOT / "data" / "Sources_clean"
CSV_PATH = SOURCES_CLEAN_DIR / "kpi_potentiel_communes.csv"
DB_DIR = Path.cwd() / "db"
DB_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DB_DIR / "parkshare_kpi.db"
SCHEMA_PATH = Path.cwd() / "sql" / "schema_kpi.sql"

print("ROOT:", ROOT)
print("CSV_PATH:", CSV_PATH)
print("DB_PATH:", DB_PATH)
print("SCHEMA_PATH:", SCHEMA_PATH)

ROOT: C:\Users\liena\Travaux\B3 - Dev\challenge48h\caca\parkshare_challenge48h_equipe9
CSV_PATH: C:\Users\liena\Travaux\B3 - Dev\challenge48h\caca\parkshare_challenge48h_equipe9\data\Sources_clean\kpi_potentiel_communes.csv
DB_PATH: c:\users\liena\Travaux\B3 - Dev\challenge48h\caca\parkshare_challenge48h_equipe9\app\db\parkshare_kpi.db
SCHEMA_PATH: c:\users\liena\Travaux\B3 - Dev\challenge48h\caca\parkshare_challenge48h_equipe9\app\sql\schema_kpi.sql


## Verification source KPI

In [2]:
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV KPI introuvable: {CSV_PATH}")

df_all = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Lignes source: {len(df_all):,}".replace(",", " "))
print(f"Colonnes ({len(df_all.columns)}):", list(df_all.columns))
df_all.head()

Lignes source: 36 012
Colonnes (25): ['code_insee', 'code_postal', 'nom_commune', 'nom_dep', 'lat', 'long', 'score_potentiel', 'rang_potentiel', 'priorite', 'score_demande', 'score_copro', 'score_parking', 'data_completeness', 'coef_completeness', 'coef_ruralite', 'population_totale', 'nb_vehicules', 'nb_vp_elec', 'nb_copros', 'nb_logements', 'grandes_copros_nb', 'part_grandes_copros', 'total_lots_stationnement', 'moyenne_lots_par_point', 'nb_points_parking']


,code_insee,code_postal,nom_commune,nom_dep,lat,long,score_potentiel,rang_potentiel,priorite,score_demande,...,population_totale,nb_vehicules,nb_vp_elec,nb_copros,nb_logements,grandes_copros_nb,part_grandes_copros,total_lots_stationnement,moyenne_lots_par_point,nb_points_parking
0,92063,92500,RUEIL MALMAISON,Hauts-de-Seine,48.876837,2.181041,99.69,1,A - tres prioritaire,0.998346,...,78186.0,251424.0,13895.0,1099.0,29623.0,181.0,16.47,7583.0,61.650407,123.0
1,93051,93160,NOISY LE GRAND,Seine-Saint-Denis,48.845703,2.552752,99.65,2,A - tres prioritaire,0.998051,...,70374.0,161633.0,17330.0,729.0,25621.0,169.0,23.18,5717.0,53.933962,106.0
2,92024,92110,CLICHY,Hauts-de-Seine,48.901996,2.306894,99.50,3,A - tres prioritaire,0.997884,...,64849.0,154386.0,8954.0,915.0,29327.0,128.0,13.99,2181.0,28.324675,77.0
3,92062,92800,PUTEAUX,Hauts-de-Seine,48.883024,2.238597,99.42,4,A - tres prioritaire,0.993565,...,43672.0,71892.0,7334.0,702.0,23393.0,106.0,15.10,5598.0,66.642857,84.0
4,78498,78300,POISSY,Yvelines,48.928040,2.041748,99.35,5,A - tres prioritaire,0.996097,...,40016.0,139516.0,31298.0,363.0,12708.0,84.0,23.14,1733.0,61.892857,28.0


## Creation du schema SQLite

In [3]:
if not SCHEMA_PATH.exists():
    raise FileNotFoundError(f"Schema SQL introuvable: {SCHEMA_PATH}")

schema_sql = SCHEMA_PATH.read_text(encoding="utf-8")

with sqlite3.connect(DB_PATH) as conn:
    conn.executescript(schema_sql)

print("Schema applique sur:", DB_PATH)

Schema applique sur: c:\users\liena\Travaux\B3 - Dev\challenge48h\caca\parkshare_challenge48h_equipe9\app\db\parkshare_kpi.db


## Chargement du CSV dans la table KPI

In [4]:
TABLE_NAME = "kpi_potentiel_communes"

df = pd.read_csv(CSV_PATH, low_memory=False)
source_rows = len(df)

numeric_cols = [
    "lat",
    "long",
    "score_potentiel",
    "rang_potentiel",
    "score_demande",
    "score_copro",
    "score_parking",
    "data_completeness",
    "coef_completeness",
    "coef_ruralite",
    "population_totale",
    "nb_vehicules",
    "nb_vp_elec",
    "nb_copros",
    "nb_logements",
    "grandes_copros_nb",
    "part_grandes_copros",
    "total_lots_stationnement",
    "moyenne_lots_par_point",
    "nb_points_parking",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["source_file"] = CSV_PATH.name

ordered_cols = [
    "code_insee",
    "code_postal",
    "nom_commune",
    "nom_dep",
    "lat",
    "long",
    "score_potentiel",
    "rang_potentiel",
    "priorite",
    "score_demande",
    "score_copro",
    "score_parking",
    "data_completeness",
    "coef_completeness",
    "coef_ruralite",
    "population_totale",
    "nb_vehicules",
    "nb_vp_elec",
    "nb_copros",
    "nb_logements",
    "grandes_copros_nb",
    "part_grandes_copros",
    "total_lots_stationnement",
    "moyenne_lots_par_point",
    "nb_points_parking",
    "source_file",
]

df_to_load = df[ordered_cols].copy()

with sqlite3.connect(DB_PATH) as conn:
    conn.execute(f"DELETE FROM {TABLE_NAME}")
    df_to_load.to_sql(TABLE_NAME, conn, if_exists="append", index=False)
    loaded_rows = conn.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}").fetchone()[0]

print(f"Lignes source: {source_rows:,}".replace(",", " "))
print(f"Lignes chargees en base: {loaded_rows:,}".replace(",", " "))
if loaded_rows != source_rows:
    raise RuntimeError("Le chargement n'est pas complet: le nombre de lignes en base differe du CSV source.")
print("Chargement complet OK (100% des lignes du CSV).")

Lignes source: 36 012
Lignes chargees en base: 36 012
Chargement complet OK (100% des lignes du CSV).


## Controle rapide apres chargement

In [5]:
with sqlite3.connect(DB_PATH) as conn:
    count_df = pd.read_sql_query(
        "SELECT COUNT(*) AS n_rows FROM kpi_potentiel_communes",
        conn,
    )
    sample_df = pd.read_sql_query(
        """
        SELECT code_insee, nom_commune, nom_dep, score_potentiel, priorite
        FROM kpi_potentiel_communes
        ORDER BY score_potentiel DESC
        LIMIT 10
        """,
        conn,
    )

count_df, sample_df

(   n_rows
 0   36012,
   code_insee             nom_commune            nom_dep  score_potentiel  \
 0      92063         RUEIL MALMAISON     Hauts-de-Seine            99.69   
 1      93051          NOISY LE GRAND  Seine-Saint-Denis            99.65   
 2      92024                  CLICHY     Hauts-de-Seine            99.50   
 3      92062                 PUTEAUX     Hauts-de-Seine            99.42   
 4      78498                  POISSY           Yvelines            99.35   
 5      60057                BEAUVAIS               Oise            99.27   
 6      78423  MONTIGNY LE BRETONNEUX           Yvelines            99.16   
 7      59378        MARCQ EN BAROEUL               Nord            98.96   
 8      78621                 TRAPPES           Yvelines            98.93   
 9      92026              COURBEVOIE     Hauts-de-Seine            98.68   
 
                priorite  
 0  A - tres prioritaire  
 1  A - tres prioritaire  
 2  A - tres prioritaire  
 3  A - tres priorit